# AI4I exploratory data analysis
This notebook is for investigation only. Production preprocessing lives in `ml/src/machineguard`. The failure-mode columns are never model features.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

ROOT = Path.cwd().parents[1] if Path.cwd().name == "notebooks" else Path.cwd()
df = pd.read_csv(ROOT / "data/raw/ai4i2020.csv")
df.shape, df.dtypes

## Quality checks
AI4I contains 10,000 observations. The following cells make missingness, duplicates, types, and class balance explicit before modeling.

In [ ]:
display(df.describe(include="all").T)
display(df.isna().sum().rename("missing"))
print("Duplicate rows:", df.duplicated().sum())
print(df["Machine failure"].value_counts(normalize=True).rename("share"))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
df["Machine failure"].value_counts().sort_index().plot.bar(ax=axes[0], title="Target counts")
df[["TWF", "HDF", "PWF", "OSF", "RNF"]].sum().sort_values().plot.barh(
    ax=axes[1], title="Failure-mode labels (EDA only)"
)
plt.tight_layout()

**Observation:** failures are rare, so accuracy is not an informative primary metric. The failure-mode flags directly identify failure mechanisms and would leak the target; they are excluded from predictors.

In [ ]:
sensor_cols = [
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]",
]
df[sensor_cols].hist(figsize=(12, 7), bins=30)
plt.suptitle("Sensor distributions")
plt.tight_layout()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(13, 7))
for column, axis in zip(sensor_cols, axes.ravel(), strict=False):
    sns.boxplot(data=df, x="Machine failure", y=column, ax=axis)
axes[-1, -1].axis("off")
plt.tight_layout()

**Observation:** several sensors overlap strongly between classes, which supports using interactions and nonlinear tree models rather than single cutoffs.

In [ ]:
corr = df[sensor_cols + ["Machine failure"]].corr()
plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="vlag", center=0)
plt.title("Sensor correlation")

In [ ]:
sns.scatterplot(
    data=df.sample(2500, random_state=42),
    x="Rotational speed [rpm]",
    y="Torque [Nm]",
    hue="Machine failure",
    alpha=0.65,
)
plt.title("Torque–speed operating envelope")

**Feature decision:** temperature difference represents thermal transfer; torque × speed is a power proxy; tool wear × torque links accumulated wear with load. Product and row identifiers are excluded because they do not describe machine condition.